# Recurrent Consensus GNN — Idea 4 Training

Loads `strict_local_negotiator_best.pt` and wraps it in `RecurrentConsensusGNN`
for K-round iterative belief refinement.

| Round | What happens |
|---|---|
| 0 | Prior = uniform over visible slots. Encode + Sinkhorn → P_0 |
| 1 | Belief features from P_0 injected. Encoder updates beliefs → P_1 |
| … | Each round, drone i sees where it 'thinks' it's going and how confident it is |
| K-1 | Final P used for Hungarian rounding → hard assignment |

Use the `torch_env` kernel.

In [ ]:
import os, random
import numpy as np
import torch

from local_negotiator import (
    COMM_RADIUS, NUM_FORMATIONS, SLOT_VISIBILITY_RADIUS, MAX_CONSENSUS_ROUNDS,
    LocalNegotiatorGNN,
    evaluate_strict_decentralized,
    load_negotiator_dataset,
    prepare_dataset,
)
from sinkhorn_head import (
    SinkhornHead, SINKHORN_ITERS, evaluate_sinkhorn,
)
from recurrent_consensus import (
    RecurrentConsensusGNN,
    NUM_ROUNDS, ROUND_EMB_DIM, RECURRENT_IN_DIM,
    evaluate_recurrent,
    train_recurrent_model,
)

SEED          = 42
DATASET       = './dataset/negotiator_dataset_v1.pt'
BASE_CKPT     = 'strict_local_negotiator_best.pt'
OUT           = 'recurrent_consensus_best_v2.pt'

EPOCHS        = 60    # less than before — more data means faster convergence
LR            = 3e-4
FREEZE_EPOCHS = 10
MAX_TRAIN     = 9000
MAX_VAL       = 500
MAX_TEST      = 500
MIN_BIJ       = 0.90
REGRET_W      = 0.40# higher than Sinkhorn run — cost awareness matters more here

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {DEVICE}  |  Torch : {torch.__version__}')
print(f'Recurrent input dim: {RECURRENT_IN_DIM} (27 original + 4 belief + 8 round)')

Device : cuda  |  Torch : 2.10.0+cu128
Recurrent input dim: 39 (27 original + 4 belief + 8 round)


In [7]:
# ── Load base checkpoint ──────────────────────────────────────────────────────
if not os.path.isfile(BASE_CKPT):
    raise FileNotFoundError(f'Missing: {BASE_CKPT}')

base_model = LocalNegotiatorGNN().to(DEVICE)
ckpt       = torch.load(BASE_CKPT, map_location='cpu', weights_only=False)
base_model.load_state_dict(ckpt['model_state_dict'])
base_model.to(DEVICE)

print('Base checkpoint loaded:')
print(f"  gossip bijection (test) : {ckpt['metrics']['test']['bijection_rate']:.3f}")
print(f"  gossip cost ratio (test): {ckpt['metrics']['test']['cost_ratio_vs_hungarian']:.4f}")
print(f"  gossip match (test)     : {ckpt['metrics']['test']['slot_match_rate']:.3f}")

# ── Dataset ───────────────────────────────────────────────────────────────────
raw_list = load_negotiator_dataset(DATASET)
train_data, val_data, test_data = prepare_dataset(
    raw_list,
    base_model.formation_embedding.weight.detach(),
    seed=SEED, force_gt_visibility=False,
)
train_data = train_data[:MAX_TRAIN]
val_data   = val_data[:MAX_VAL]
test_data  = test_data[:MAX_TEST]
print(f'\nTrain={len(train_data)}  Val={len(val_data)}  Test={len(test_data)}')

Base checkpoint loaded:
  gossip bijection (test) : 0.540
  gossip cost ratio (test): 1.0417
  gossip match (test)     : 0.587

Train=6000  Val=500  Test=500


In [8]:
# ── Build RecurrentConsensusGNN ───────────────────────────────────────────────
sinkhorn_head = SinkhornHead(n_iters=SINKHORN_ITERS).to(DEVICE)
model = RecurrentConsensusGNN(
    base_model    = base_model,
    sinkhorn_head = sinkhorn_head,
    num_rounds    = NUM_ROUNDS,
    round_emb_dim = ROUND_EMB_DIM,
).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
new_params   = sum(p.numel() for p in [
    *model.round_embedding.parameters(),
    *model.recurrent_proj.parameters(),
    *model.sinkhorn_head.parameters(),
])
print(f'Total params  : {total_params:,}')
print(f'New params    : {new_params:,}  (frozen encoder trains {total_params - new_params:,})')
print(f'Rounds K      : {NUM_ROUNDS} (annealed from 2 → {NUM_ROUNDS} over first 20 epochs)')
print(f'Freeze epochs : {FREEZE_EPOCHS}')

Total params  : 33,283
New params    : 2,657  (frozen encoder trains 30,626)
Rounds K      : 4 (annealed from 2 → 4 over first 20 epochs)
Freeze epochs : 10


In [9]:
# ── Baselines on test set (before any recurrent training) ─────────────────────
print('=== Baselines (test set) ===')

print('\nGossip consensus (original):')
gossip = evaluate_strict_decentralized(base_model, test_data, DEVICE)
for k, v in gossip.items(): print(f'  {k}: {v:.4f}')

print('\nSinkhorn single-pass (untrained head, base encoder):')
sink_baseline = evaluate_sinkhorn(base_model, sinkhorn_head, test_data, DEVICE)
for k, v in sink_baseline.items(): print(f'  {k}: {v:.4f}')

print('\nRecurrent K=4 (untrained, base encoder):')
recur_baseline = evaluate_recurrent(model, test_data, DEVICE, max_rounds=NUM_ROUNDS)
for k, v in recur_baseline.items(): print(f'  {k}: {v:.4f}')

=== Baselines (test set) ===

Gossip consensus (original):
  bijection_rate: 0.5400
  conflict_rate: 0.0298
  unassigned_rate: 0.0155
  cost_ratio_vs_hungarian: 1.0417
  slot_match_rate: 0.5866
  consensus_rounds: 9.6520
  converged_rate: 0.5480

Sinkhorn single-pass (untrained head, base encoder):
  bijection_rate: 1.0000
  conflict_rate: 0.0000
  unassigned_rate: 0.0000
  cost_ratio_vs_hungarian: 1.0152
  slot_match_rate: 0.6941
  consensus_rounds: 0.0000
  converged_rate: 1.0000

Recurrent K=4 (untrained, base encoder):
  bijection_rate: 1.0000
  conflict_rate: 0.0000
  unassigned_rate: 0.0000
  cost_ratio_vs_hungarian: 1.0153
  slot_match_rate: 0.6931
  consensus_rounds: 2.5180
  converged_rate: 0.7800
  avg_rounds_used: 2.5180


In [10]:
# ── Train ─────────────────────────────────────────────────────────────────────
print('=== Recurrent consensus training ===')
history = train_recurrent_model(
    model, train_data, val_data, DEVICE,
    epochs        = EPOCHS,
    lr            = LR,
    freeze_epochs = FREEZE_EPOCHS,
    min_bijection_for_best = MIN_BIJ,
    force_gt_train = True,
    regret_weight  = REGRET_W,
)

=== Recurrent consensus training ===
Epoch   1 [frozen|K=2] | train=0.5317 | val=0.6904 | bij=1.000 | cost=1.0148 | match=0.681 | rounds=2.0
Epoch   5 [frozen|K=2] | train=0.4946 | val=0.6761 | bij=1.000 | cost=1.0148 | match=0.702 | rounds=2.0
Epoch  10 [frozen|K=3] | train=0.4871 | val=0.6896 | bij=1.000 | cost=1.0167 | match=0.703 | rounds=2.5
  [Epoch 11] Encoder unfrozen — joint fine-tuning (K=3).
Epoch  15 [joint|K=3] | train=0.4305 | val=0.6483 | bij=1.000 | cost=1.0143 | match=0.710 | rounds=2.7
Epoch  20 [joint|K=4] | train=0.4261 | val=0.6580 | bij=1.000 | cost=1.0135 | match=0.713 | rounds=3.1
Epoch  25 [joint|K=4] | train=0.4161 | val=0.6732 | bij=1.000 | cost=1.0135 | match=0.728 | rounds=3.1
Epoch  30 [joint|K=4] | train=0.4100 | val=0.6776 | bij=1.000 | cost=1.0139 | match=0.744 | rounds=3.0
Epoch  35 [joint|K=4] | train=0.4019 | val=0.7022 | bij=1.000 | cost=1.0135 | match=0.741 | rounds=3.0
Epoch  40 [joint|K=4] | train=0.3967 | val=0.7062 | bij=1.000 | cost=1.0136 | m

KeyboardInterrupt: 

In [12]:
# Emergency save — run this now before doing anything else
import torch

torch.save({
    'model_state_dict': model.state_dict(),
    'config': {
        'num_rounds':             model.num_rounds,
        'round_emb_dim':          ROUND_EMB_DIM,
        'sinkhorn_iters':         SINKHORN_ITERS,
        'recurrent_in_dim':       RECURRENT_IN_DIM,
        'comm_radius':            COMM_RADIUS,
        'slot_visibility_radius': SLOT_VISIBILITY_RADIUS,
        'num_formations':         NUM_FORMATIONS,
    },
    'epoch':   70,
    
}, 'recurrent_consensus_ep70.pt')

print('Saved recurrent_consensus_ep70.pt')

Saved recurrent_consensus_ep70.pt


In [ ]:
# ── Final evaluation — all three inference modes on same test set ──────────────
print('=== Final evaluation (test set) ===')

recur_fixed  = evaluate_recurrent(model, test_data, DEVICE,
                                   max_rounds=NUM_ROUNDS, early_stop=False)
recur_early  = evaluate_recurrent(model, test_data, DEVICE,
                                   max_rounds=NUM_ROUNDS, early_stop=True)
sink_final   = evaluate_sinkhorn(model.formation_embedding.__class__,
                                  sinkhorn_head, test_data, DEVICE)

keys = ['bijection_rate','cost_ratio_vs_hungarian','slot_match_rate',
        'conflict_rate','unassigned_rate','avg_rounds_used']

print(f'\n{"metric":<35} {"gossip":>10} {"recur-fixed":>12} {"recur-early":>12}')
print('-' * 72)
for k in keys:
    g  = gossip.get(k, 0.0)
    rf = recur_fixed.get(k, 0.0)
    re = recur_early.get(k, 0.0)
    print(f'{k:<35} {g:>10.4f} {rf:>12.4f} {re:>12.4f}')

In [ ]:
# ── Save ──────────────────────────────────────────────────────────────────────
torch.save({
    'model_state_dict': model.state_dict(),
    'config': {
        'num_formations':         NUM_FORMATIONS,
        'comm_radius':            COMM_RADIUS,
        'slot_visibility_radius': SLOT_VISIBILITY_RADIUS,
        'num_rounds':             NUM_ROUNDS,
        'round_emb_dim':          ROUND_EMB_DIM,
        'sinkhorn_iters':         SINKHORN_ITERS,
        'base_checkpoint':        BASE_CKPT,
        'recurrent_in_dim':       RECURRENT_IN_DIM,
    },
    'history': history,
    'metrics': {
        'recurrent_fixed': recur_fixed,
        'recurrent_early': recur_early,
        'gossip_baseline': gossip,
    },
}, OUT)
print(f'Saved → {OUT}')

In [ ]:
# ── Training curves ───────────────────────────────────────────────────────────
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 4, figsize=(20, 4))
fig.suptitle('Recurrent Consensus Training', fontsize=13, fontweight='bold')
ep = range(1, len(history['train_loss']) + 1)

axes[0].plot(ep, history['train_loss'], label='train', color='steelblue')
axes[0].plot(ep, history['val_loss'],   label='val',   color='coral', linestyle='--')
axes[0].axvline(FREEZE_EPOCHS, color='gray', linestyle=':', alpha=0.6, label='unfreeze')
axes[0].set_title('Loss'); axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(ep, history['val_bijection_rate'], color='green')
axes[1].axhline(0.90, color='green', linestyle=':', alpha=0.5)
axes[1].axvline(FREEZE_EPOCHS, color='gray', linestyle=':', alpha=0.6)
axes[1].set_title('Val Bijection Rate'); axes[1].grid(alpha=0.3)

axes[2].plot(ep, history['val_cost_ratio_vs_hungarian'], color='orange')
axes[2].axhline(1.0, color='gray', linestyle='-', alpha=0.3)
axes[2].axvline(FREEZE_EPOCHS, color='gray', linestyle=':', alpha=0.6)
axes[2].set_title('Cost Ratio vs Hungarian'); axes[2].grid(alpha=0.3)

axes[3].plot(ep, history['val_avg_rounds_used'], color='purple')
axes[3].axhline(NUM_ROUNDS, color='purple', linestyle=':', alpha=0.4, label=f'max K={NUM_ROUNDS}')
axes[3].set_title('Avg Rounds Used (early stop)'); axes[3].legend(); axes[3].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('recurrent_training_curves.png', dpi=120)
plt.show()

In [14]:
import time, torch
from local_negotiator import (
    build_candidate_mask,
    build_candidate_edges,
    build_candidate_edge_attr,
    build_comm_graph,
    build_node_features,
    SLOT_VISIBILITY_RADIUS,
)
from recurrent_consensus import ROUND_EMB_DIM, RECURRENT_IN_DIM
from sinkhorn_head import SINKHORN_ITERS
from local_negotiator import COMM_RADIUS, NUM_FORMATIONS
# Time one epoch manually to find the bottleneck
model.eval()
times = {'mask': 0, 'features': 0, 'comm': 0, 'edges': 0, 'forward': 0}
N_PROFILE = 50

with torch.no_grad():
    for data in train_data[:N_PROFILE]:
        t0 = time.time()
        mask = build_candidate_mask(data.drone_pos, data.slots,
                                     SLOT_VISIBILITY_RADIUS, y=data.y, force_gt=True)
        times['mask'] += time.time() - t0

        t0 = time.time()
        fid   = data.formation_id.item()
        f_emb = model.formation_embedding(torch.tensor(fid, device=DEVICE))
        x     = build_node_features(data.drone_pos, data.slots,
                                     f_emb.detach().cpu(), candidate_mask=mask)
        times['features'] += time.time() - t0

        t0 = time.time()
        ei, ea = build_comm_graph(data.drone_pos)
        times['comm'] += time.time() - t0

        t0 = time.time()
        ds_ei = build_candidate_edges(mask)[0]
        ds_ea = build_candidate_edge_attr(data.drone_pos, data.slots, ds_ei)
        times['edges'] += time.time() - t0

        t0 = time.time()
        Ps, _ = model.forward_recurrent(
            x.to(DEVICE), data.slots.to(DEVICE), mask.to(DEVICE),
            ei.to(DEVICE), ea.to(DEVICE),
            ds_ei.to(DEVICE), ds_ea.to(DEVICE), K=4
        )
        times['forward'] += time.time() - t0

total_profiled = sum(times.values())
print(f'Over {N_PROFILE} samples:')
for k, v in times.items():
    pct = 100 * v / total_profiled
    per_sample = 1000 * v / N_PROFILE
    print(f'  {k:12s}: {per_sample:6.1f} ms/sample  ({pct:.1f}%)')

per_epoch_est = total_profiled / N_PROFILE * len(train_data) / 60
print(f'\nEstimated time per epoch: {per_epoch_est:.1f} min')
print(f'Estimated 80 epochs: {per_epoch_est * 80 / 60:.1f} hours')

Over 50 samples:
  mask        :    0.1 ms/sample  (0.2%)
  features    :    0.8 ms/sample  (1.2%)
  comm        :    1.3 ms/sample  (2.0%)
  edges       :    0.8 ms/sample  (1.2%)
  forward     :   63.9 ms/sample  (95.4%)

Estimated time per epoch: 6.7 min
Estimated 80 epochs: 8.9 hours


In [ ]:
import torch
from recurrent_consensus import RecurrentConsensusGNN, evaluate_recurrent, NUM_ROUNDS, ROUND_EMB_DIM
from sinkhorn_head import SinkhornHead, SINKHORN_ITERS, evaluate_sinkhorn
from local_negotiator import (
    LocalNegotiatorGNN, evaluate_strict_decentralized,
    load_negotiator_dataset, prepare_dataset, COMM_RADIUS,
    SLOT_VISIBILITY_RADIUS, NUM_FORMATIONS
)

# Load checkpoint
ckpt       = torch.load('recurrent_consensus_ep70.pt', weights_only=False, map_location='cpu')
base_model = LocalNegotiatorGNN()
sh         = SinkhornHead(n_iters=SINKHORN_ITERS)
model      = RecurrentConsensusGNN(base_model, sh, num_rounds=NUM_ROUNDS, round_emb_dim=ROUND_EMB_DIM)
model.load_state_dict(ckpt['model_state_dict'])
model.to(DEVICE)
model.eval()

# Load test data using the model's trained formation embeddings
raw_list = load_negotiator_dataset('./dataset/negotiator_dataset_v1.pt')
_, _, test_data = prepare_dataset(
    raw_list,
    model.formation_embedding.weight.detach(),
    seed=42, force_gt_visibility=False
)
test_data = test_data[:500]

# Run all three inference modes on same test set
print('=== Recurrent K=4 fixed ===')
m1 = evaluate_recurrent(model, test_data, DEVICE, max_rounds=4, early_stop=False)
for k,v in m1.items(): print(f'  {k}: {v:.4f}')

print('\n=== Recurrent early stop ===')
m2 = evaluate_recurrent(model, test_data, DEVICE, max_rounds=4, early_stop=True)
for k,v in m2.items(): print(f'  {k}: {v:.4f}')

print('\n=== Sinkhorn single-pass (same weights) ===')


=== Recurrent K=4 fixed ===
  bijection_rate: 1.0000
  conflict_rate: 0.0000
  unassigned_rate: 0.0000
  cost_ratio_vs_hungarian: 1.0121
  slot_match_rate: 0.7520
  consensus_rounds: 4.0000
  converged_rate: 0.0000
  avg_rounds_used: 4.0000

=== Recurrent early stop ===
  bijection_rate: 1.0000
  conflict_rate: 0.0000
  unassigned_rate: 0.0000
  cost_ratio_vs_hungarian: 1.0117
  slot_match_rate: 0.7526
  consensus_rounds: 3.0180
  converged_rate: 0.6100
  avg_rounds_used: 3.0180

=== Sinkhorn single-pass (same weights) ===


TypeError: Module.eval() missing 1 required positional argument: 'self'

In [19]:
@torch.no_grad()
def evaluate_sinkhorn_on_recurrent(model, dataset, device,
                                    slot_radius=SLOT_VISIBILITY_RADIUS):
    model.eval()
    results = {
        'bijection_rate': [], 'conflict_rate': [],
        'unassigned_rate': [], 'cost_ratio_vs_hungarian': [],
        'slot_match_rate': [], 'consensus_rounds': [],
        'converged_rate': [], 'avg_rounds_used': [],
    }
    for data in dataset:
        fid   = data.formation_id.item()
        f_emb = model.formation_embedding(torch.tensor(fid, device=device))
        dp, sl, y = data.drone_pos, data.slots, data.y
        N = dp.size(0)

        mask  = build_candidate_mask(dp, sl, slot_radius, y=y, force_gt=False)
        x     = build_node_features(dp, sl, f_emb.detach().cpu(),
                                     candidate_mask=mask).to(device)
        ei, ea = build_comm_graph(dp)
        ds_ei  = build_candidate_edges(mask)[0]
        ds_ea  = build_candidate_edge_attr(dp, sl, ds_ei)

        # Pad x to 39-dim: append zeros for belief (4) and round (8) features
        # This is equivalent to round 0 with uniform prior and round_emb zeroed
        padding = torch.zeros(N, BELIEF_FEAT_DIM + ROUND_EMB_DIM, device=device)
        x_padded = torch.cat([x, padding], dim=1)   # (N, 39)

        # Single pass through recurrent encoder
        from local_negotiator import edge_logits_to_dense
        h      = model._encode(x_padded, ei.to(device), ea.to(device))
        logits = model._score_edges(h, ds_ei.to(device), ds_ea.to(device))
        dense  = edge_logits_to_dense(logits, ds_ei.to(device), N, fill=-1e4)
        dense  = dense.masked_fill(~mask.to(device), -1e4)
        P      = model.sinkhorn_head(dense, mask.to(device))

        # Hungarian rounding
        row_ind, col_ind = linear_sum_assignment(-P.cpu().numpy())
        assignment = torch.full((N,), -1, dtype=torch.long)
        assignment[row_ind] = torch.tensor(col_ind, dtype=torch.long)

        m = assignment_quality_metrics(dp, sl, assignment, y, rounds=0.0)
        for k, v in m.items():
            results[k].append(v)
        results['converged_rate'].append(1.0)
        results['avg_rounds_used'].append(0.0)

    return {k: float(np.mean(v)) for k, v in results.items()}


# Make sure BELIEF_FEAT_DIM and ROUND_EMB_DIM are imported
from recurrent_consensus import BELIEF_FEAT_DIM, ROUND_EMB_DIM

m3 = evaluate_sinkhorn_on_recurrent(model, test_data, DEVICE)
print('\n=== Sinkhorn single-pass (recurrent encoder, no rounds) ===')
for k, v in m3.items(): print(f'  {k}: {v:.4f}')

# Full comparison table
print('\n=== Head-to-head (test set) ===')
keys = ['bijection_rate', 'cost_ratio_vs_hungarian',
        'slot_match_rate', 'conflict_rate', 'avg_rounds_used']
print(f'{"metric":<35} {"gossip":>10} {"sinkhorn-1pass":>14} {"recur-early":>12}')
print('-' * 74)
gossip = {
    'bijection_rate':          0.540,
    'cost_ratio_vs_hungarian': 1.0417,
    'slot_match_rate':         0.5866,
    'conflict_rate':           0.0298,
    'avg_rounds_used':         9.652,
}
for k in keys:
    g = gossip.get(k, 0.0)
    s = m3.get(k, 0.0)
    r = m2.get(k, 0.0)
    better = ('↑' if k not in ('cost_ratio_vs_hungarian', 'conflict_rate',
                                'avg_rounds_used') and r > s
              else ('↓' if k in ('cost_ratio_vs_hungarian', 'conflict_rate')
                    and r < s else ' '))
    print(f'{k:<35} {g:>10.4f} {s:>14.4f} {r:>12.4f} {better}')


=== Sinkhorn single-pass (recurrent encoder, no rounds) ===
  bijection_rate: 1.0000
  conflict_rate: 0.0000
  unassigned_rate: 0.0000
  cost_ratio_vs_hungarian: 1.0193
  slot_match_rate: 0.7070
  consensus_rounds: 0.0000
  converged_rate: 1.0000
  avg_rounds_used: 0.0000

=== Head-to-head (test set) ===
metric                                  gossip sinkhorn-1pass  recur-early
--------------------------------------------------------------------------
bijection_rate                          0.5400         1.0000       1.0000  
cost_ratio_vs_hungarian                 1.0417         1.0193       1.0117 ↓
slot_match_rate                         0.5866         0.7070       0.7526 ↑
conflict_rate                           0.0298         0.0000       0.0000  
avg_rounds_used                         9.6520         0.0000       3.0180  
